In [0]:
%pip install pendulum

In [0]:
import requests
import json
import pendulum
from pathlib import Path
import pandas as pd


In [0]:
yesterday = pendulum.now("America/Bogota").subtract(days=1).to_date_string()
print(yesterday)

In [0]:
# Detecta el catálogo actual (suele ser 'workspace' o 'main')
current_catalog = spark.sql("select current_catalog()").first()[0]
catalog = current_catalog
schema = "raw"
volume = "uses"

# Crea esquema y volumen si no existen
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

In [0]:
# Elimina todos los archivos y subcarpetas dentro de la ruta
dbutils.fs.rm("/Volumes/workspace/raw/uses", recurse=True)

In [0]:
# Crear widgets
dbutils.widgets.text("date_ini", yesterday, "Start Date (YYYY-MM-DD)")
#dbutils.widgets.text("date_fin", "2023-12-31", "End Date (YYYY-MM-DD)")
dbutils.widgets.text("output_dir", "/Volumes/workspace/raw/uses", "Output Directory")
dbutils.widgets.text("trx_id","99999999999","Transaction ID")
dbutils.widgets.text("enpoint", "https://api.trafpay.cl/api-mifare/transactions/GetTransacionByID?ProjectID=1&TransactionID={trx_id}", "API endpoint")
dbutils.widgets.text("timeout", "60", "Time Out")

# Leer valores de los widgets
start_date = pendulum.parse(dbutils.widgets.get("date_ini")).date()
#end_date = pendulum.parse(dbutils.widgets.get("end_date")).date()
output_dir = Path(dbutils.widgets.get("output_dir"))
trx_id = dbutils.widgets.get("trx_id")
endpoint_dir = dbutils.widgets.get("enpoint")
API_URL = endpoint_dir
timeout = int(dbutils.widgets.get("timeout"))

In [0]:
saved_files = []
session = requests.Session()

current_date = pendulum.parse(dbutils.widgets.get("date_ini"))

daily_output_dir = (
    output_dir
    / f"{current_date.year:04d}"
    / f"{current_date.month:02d}"
    / f"{current_date.day:02d}"
)

daily_output_dir.mkdir(parents=True, exist_ok=True)

file_path = daily_output_dir / "uses.json"

response = session.get(API_URL, timeout=timeout)

if response.status_code != 200:
    raise Exception(f"Error en API: {response.status_code} - {response.text}")

# Guardar directamente el JSON
file_path.write_bytes(response.content)

# Guardar ruta en lista
saved_files.append(str(file_path))

current_date = current_date.add(days=1)
print(f"Archivos guardados: {len(saved_files)}")

In [0]:
print(daily_output_dir)

In [0]:
import requests
import pandas as pd
api_key = dbutils.secrets.get(scope="key-apis", key="usosapikey")
url = "https://api.trafpay.cl/api-mifare/transactions?ProjectID=1&FechaInicio=2026-01-01&FechaFinal=2026-01-01"
headers = {"apikey": api_key}

r = requests.get(url, headers=headers, timeout=30)
print(r.status_code)

print (r.text)